# Laplace Approximation: Stochastic Volatility

This example is part of the *Universal Inference* section, which explores running inference in a broad class of models by approximating every continuous latent variable with a Gaussian. The first example, *Laplace Approximation: Logistic Regression*, introduced a small Laplace toolkit; here we apply the same toolkit to a time series.

The **stochastic volatility** model is a standard benchmark for approximate inference in nonlinear state-space models (see Durbin and Koopman, *Time Series Analysis by State Space Methods*, and Martino et al., 2011, who use it to evaluate INLA). The log-variance of a series of returns follows a hidden autoregressive process:

$$\begin{aligned}
x_1 &\sim \mathcal{N}\left(0, \frac{\sigma^2}{1-\varphi^2}\right),\\
x_t &\sim \mathcal{N}\left(\varphi x_{t-1}, \sigma^2\right),\\
y_t &\sim \mathcal{N}\left(0, \exp(\mu + x_t)\right).
\end{aligned}$$

The transition is linear and Gaussian, so RxInfer handles it with exact rules. The observation is not: the latent variable sits inside the variance through an exponential, and no conjugate rule exists. We give the observation a fused node with a `logpdf`, let the rule fallback turn it into messages, and use `LaplaceForm` to keep every message on the chain Gaussian. The result is a Laplace version of the Kalman smoother, obtained without deriving a single update rule.

!!! note
    The Laplace machinery below lives in this notebook on purpose. It is a preview of an upcoming generic approximation layer (ExpectationApproximations.jl and a universal rules package), and will move there once it matures.

In [ ]:
using RxInfer, ReactiveMP, BayesBase, Distributions, LinearAlgebra
using ForwardDiff, StatsFuns, StableRNGs, Plots

## The Laplace toolkit

The Laplace approximation of a density $p(x) \propto \exp(\ell(x))$ is the Gaussian

$$\mathcal{N}\left(x \mid \hat{x}, \left(-\nabla^2 \ell(\hat{x})\right)^{-1}\right),\qquad \hat{x} = \arg\max_x \ell(x).$$

The function `laplace` finds the mode $\hat{x}$ with a damped Newton method and returns the Gaussian. Gradients and Hessians come from automatic differentiation (ForwardDiff.jl), so $\ell$ can be any Julia function.

In [ ]:
function laplace(logp, x0::Real; tol = 1e-10, maxiters = 100)
    x = float(x0)
    d2(z) = ForwardDiff.derivative(w -> ForwardDiff.derivative(logp, w), z)
    for _ in 1:maxiters
        g, h = ForwardDiff.derivative(logp, x), d2(x)
        h < 0 || error("Laplace: the log-density is not concave at x = $x")
        step = -g / h
        t, fx = 1.0, logp(x)
        while logp(x + t * step) < fx - 1e-12 && t > 1e-8 # backtracking line search
            t /= 2
        end
        x += t * step
        abs(t * step) < tol && break
    end
    return NormalMeanVariance(x, -inv(d2(x)))
end

function laplace(logp, x0::AbstractVector; tol = 1e-10, maxiters = 100)
    x = float.(collect(x0))
    for _ in 1:maxiters
        g, H = ForwardDiff.gradient(logp, x), ForwardDiff.hessian(logp, x)
        F = cholesky(Symmetric(-H); check = false)
        issuccess(F) || error("Laplace: the log-density is not concave at x = $x")
        step = F \ g
        t, fx = 1.0, logp(x)
        while logp(x + t * step) < fx - 1e-12 && t > 1e-8 # backtracking line search
            t /= 2
        end
        x += t * step
        norm(t * step) < tol && break
    end
    return MvNormalMeanCovariance(x, Matrix(Symmetric(inv(-ForwardDiff.hessian(logp, x)))))
end

Next, the form constraint. In RxInfer a [form constraint](https://reactivebayes.github.io/ReactiveMP.jl/stable/lib/form-constraints/) is applied to the product of the messages that arrive at a variable. When no closed-form product exists, RxInfer keeps that product as a lazy `ProductOf` object which only knows how to evaluate its `logpdf`; `LaplaceForm` replaces it with its Laplace approximation. Gaussians pass through untouched, so conjugate parts of a model keep their exact updates. The Newton search starts from the mean of the first Gaussian factor in the product, usually the prior, or from zero if the product has no Gaussian factor.

In [ ]:
struct LaplaceForm <: ReactiveMP.AbstractFormConstraint end

ReactiveMP.default_form_check_strategy(::LaplaceForm) = FormConstraintCheckLast()
ReactiveMP.default_prod_constraint(::LaplaceForm) = GenericProd()

ReactiveMP.constrain_form(::LaplaceForm, d::NormalDistributionsFamily) = d
function ReactiveMP.constrain_form(::LaplaceForm, d)
    # A product without a Gaussian factor (a likelihood alone) starts from zero, scalars only
    x0 = something(startpoint(d), 0.0)
    return laplace(x -> logpdf(d, x), x0)
end

startpoint(d::NormalDistributionsFamily) = mean(d)
startpoint(d::ProductOf) = firstpoint((d.left, d.right))
startpoint(d::LinearizedProductOf) = firstpoint(d.vector)
startpoint(d) = nothing

function firstpoint(factors)
    for factor in factors
        x0 = startpoint(factor)
        isnothing(x0) || return x0
    end
    return nothing
end

## Data

We simulate $T = 200$ returns with a persistent volatility process. The parameters $\mu$, $\varphi$ and $\sigma$ are known in this example; only the volatility path $x_{1:T}$ is inferred.

In [ ]:
rng = StableRNG(1234)
T, μ, φ, σ = 200, -1.0, 0.95, 0.3

x_true = zeros(T)
x_true[1] = rand(rng, Normal(0, σ / sqrt(1 - φ^2)))
for t in 2:T
    x_true[t] = φ * x_true[t-1] + σ * randn(rng)
end
y = [exp((μ + x_true[t]) / 2) * randn(rng) for t in 1:T]

p1 = plot(y; label = "returns yₜ", color = :gray, xlabel = "t")
p2 = plot(x_true; label = "log-volatility xₜ", color = :black, xlabel = "t")
plot(p1, p2; layout = (2, 1), size = (800, 450))

## The model

The observation node evaluates $\log \mathcal{N}(y_t \mid 0, e^{\mu + x_t})$. Its interfaces are the observation `out`, the latent `x` and the constant `μ`.

In [ ]:
struct SVObservation{X, M} <: ContinuousUnivariateDistribution
    x::X
    μ::M
end

BayesBase.logpdf(d::SVObservation, y) = (v = d.μ + d.x; -(log2π + v + y^2 * exp(-v)) / 2)
BayesBase.insupport(::SVObservation, y) = true

@define_factor_node(node = SVObservation, type = Stochastic, interfaces = [:out, :x, :μ])

In [ ]:
@model function stochastic_volatility(y, μ, φ, σ)
    x[1] ~ NormalMeanVariance(0.0, σ^2 / (1 - φ^2))
    y[1] ~ SVObservation(x[1], μ)
    for t in 2:length(y)
        x[t] ~ NormalMeanVariance(φ * x[t-1], σ^2)
        y[t] ~ SVObservation(x[t], μ)
    end
end

Under belief propagation, a message that leaves $x_t$ along the chain is the product of the message from the other side of the chain and the observation message. That product is not Gaussian, so the constraint is placed on the messages, `μ(x) :: LaplaceForm()`, as well as on the marginals, `q(x) :: LaplaceForm()`. This keeps the transition nodes fed with Gaussian messages, for which exact rules exist.

In [ ]:
laplace_constraints = @constraints begin
    μ(x)::LaplaceForm()
    q(x)::LaplaceForm()
end

result = infer(
    model = stochastic_volatility(μ = μ, φ = φ, σ = σ),
    data = (y = y,),
    constraints = laplace_constraints,
    options = (rulefallback = NodeFunctionRuleFallback(),),
)

q_x = result.posteriors[:x]
m_mp, s_mp = mean.(q_x), std.(q_x)

## Sanity check: a linear-Gaussian observation

If the observation is Gaussian, $y_t \sim \mathcal{N}(x_t, r)$, every product is Gaussian and the Laplace approximation is exact. The same machinery must then reproduce the Kalman smoother that RxInfer computes with its exact rules. We write the Gaussian observation as a fused node, so that it goes through the fallback and `LaplaceForm` like the stochastic volatility observation.

In [ ]:
struct GaussianObservation{X, R} <: ContinuousUnivariateDistribution
    x::X
    r::R
end

BayesBase.logpdf(d::GaussianObservation, y) = -(log2π + log(d.r) + (y - d.x)^2 / d.r) / 2
BayesBase.insupport(::GaussianObservation, y) = true

@define_factor_node(node = GaussianObservation, type = Stochastic, interfaces = [:out, :x, :r])

@model function linear_gaussian(y, r, φ, σ, fused)
    x[1] ~ NormalMeanVariance(0.0, σ^2 / (1 - φ^2))
    for t in 1:length(y)
        if t > 1
            x[t] ~ NormalMeanVariance(φ * x[t-1], σ^2)
        end
        if fused
            y[t] ~ GaussianObservation(x[t], r)
        else
            y[t] ~ NormalMeanVariance(x[t], r)
        end
    end
end

y_linear = x_true .+ sqrt(0.5) .* randn(rng, T)

result_fused = infer(
    model = linear_gaussian(r = 0.5, φ = φ, σ = σ, fused = true),
    data = (y = y_linear,),
    constraints = laplace_constraints,
    options = (rulefallback = NodeFunctionRuleFallback(),),
)
result_exact = infer(model = linear_gaussian(r = 0.5, φ = φ, σ = σ, fused = false), data = (y = y_linear,))

println("max |mean difference|     = ", maximum(abs, mean.(result_fused.posteriors[:x]) - mean.(result_exact.posteriors[:x])))
println("max |variance difference| = ", maximum(abs, var.(result_fused.posteriors[:x]) - var.(result_exact.posteriors[:x])))

## References for the nonlinear model

We compare the message passing result with two references.

1. **Global Laplace approximation.** The Laplace approximation of the joint posterior $p(x_{1:T} \mid y_{1:T})$, computed directly by Newton's method on the full $T$-dimensional log-density. Its marginals are what an INLA-style Gaussian approximation reports. Our message passing scheme projects each message separately, so the two need not coincide.
2. **Elliptical slice sampling** (Murray, Adams and MacKay, 2010), a simple MCMC method that applies whenever the prior is Gaussian, as it is here. A long run gives the "exact" posterior marginals.

In [ ]:
function logprior(x)
    s = logpdf(Normal(0, σ / sqrt(1 - φ^2)), x[1])
    for t in 2:T
        s += logpdf(Normal(φ * x[t-1], σ), x[t])
    end
    return s
end

loglikelihood(x) = sum(logpdf(SVObservation(x[t], μ), y[t]) for t in 1:T)

q_global = laplace(x -> logprior(x) + loglikelihood(x), zeros(T))
m_global, s_global = mean(q_global), sqrt.(diag(cov(q_global)))

In [ ]:
function prior_draw(rng)
    x = zeros(T)
    x[1] = σ / sqrt(1 - φ^2) * randn(rng)
    for t in 2:T
        x[t] = φ * x[t-1] + σ * randn(rng)
    end
    return x
end

function elliptical_slice_sampling(rng, x0; nsamples, burnin)
    x, ℓ = copy(x0), loglikelihood(x0)
    s1, s2 = zeros(T), zeros(T)
    for iteration in 1:(nsamples + burnin)
        ν = prior_draw(rng)
        threshold = ℓ + log(rand(rng))
        θ = 2π * rand(rng)
        lo, hi = θ - 2π, θ
        while true
            proposal = x .* cos(θ) .+ ν .* sin(θ)
            ℓproposal = loglikelihood(proposal)
            if ℓproposal > threshold
                x, ℓ = proposal, ℓproposal
                break
            end
            θ < 0 ? (lo = θ) : (hi = θ)
            θ = lo + (hi - lo) * rand(rng)
        end
        if iteration > burnin
            s1 .+= x
            s2 .+= x .^ 2
        end
    end
    m = s1 ./ nsamples
    return m, sqrt.(s2 ./ nsamples .- m .^ 2)
end

m_ess, s_ess = elliptical_slice_sampling(StableRNG(7), m_global; nsamples = 50_000, burnin = 5_000)

## Results

In [ ]:
ts = 1:T
plt = plot(ts, m_ess; ribbon = 2s_ess, fillalpha = 0.25, color = :gray, label = "slice sampler (±2 sd)", xlabel = "t", ylabel = "xₜ", size = (900, 500), legend = :outerbottom, legendcolumns = 2)
plot!(plt, ts, m_global; color = :blue, label = "global Laplace")
plot!(plt, ts, m_mp; ribbon = 2s_mp, fillalpha = 0.15, color = :red, linestyle = :dash, label = "Laplace message passing (±2 sd)")
plot!(plt, ts, x_true; color = :black, linewidth = 1, label = "true xₜ")

In [ ]:
rmse(a, b) = sqrt(sum(abs2, a - b) / length(a))

println(rpad("", 26), rpad("RMSE of means", 16), "RMSE of sds")
println(rpad("message passing vs ESS", 26), rpad(round(rmse(m_mp, m_ess); digits = 3), 16), round(rmse(s_mp, s_ess); digits = 3))
println(rpad("global Laplace vs ESS", 26), rpad(round(rmse(m_global, m_ess); digits = 3), 16), round(rmse(s_global, s_ess); digits = 3))
println(rpad("message passing vs global", 26), rpad(round(rmse(m_mp, m_global); digits = 3), 16), round(rmse(s_mp, s_global); digits = 3))
println()
println("average posterior sd: message passing = ", round(sum(s_mp) / T; digits = 3), ", global Laplace = ", round(sum(s_global) / T; digits = 3), ", ESS = ", round(sum(s_ess) / T; digits = 3))

## Exact vs approximate, and what did not work

- **What worked.** The same 40 lines of Laplace machinery that fitted a logistic regression now run a nonlinear state-space smoother. On a linear-Gaussian model they reproduce RxInfer's exact Kalman smoother to rounding error, which is the property we want from a universal method: no loss where exact inference is possible.
- **Accuracy.** The posterior standard deviations from message passing are close to the sampler's. The means follow the sampler less closely than the global Laplace approximation does: each message is projected to a Gaussian on its own, at the mode of a *local* product, so errors made early in the forward pass are not corrected later. Both Laplace approximations also sit slightly below the sampler's mean throughout: the posterior of the log-volatility is skewed, and a Laplace approximation reports its mode rather than its mean.
- **What is missing.** True *Laplace propagation* (Eskin, Smola and Vishwanathan, 2003) projects the *marginal* and divides out the incoming message, like expectation propagation, and iterates to a fixed point that coincides with the global posterior mode. Our scheme computes each message once; additional iterations change nothing. Implementing the cavity-based variant requires access to the incoming message inside the projection, which is planned for the generic layer.
- **What we had to work around.** As in the first example, the observation is a fused node, the messages need an explicit `μ(x) :: LaplaceForm()` constraint, and the free energy is not available because the fallback messages are unnormalised. The parameters $\mu$, $\varphi$ and $\sigma$ are fixed; learning them would add a Gaussian-in-log-space parameter to every observation node, which is the subject of the next example.